# Flattening Semi-structured Data

This notebook uses Snowpark (Python API) to flatten the semi-structured data in the VARIANT column of a Snowflake table. Snowpark supports two different general programming techniques to manipulate data:

* **SQL programming**, which lets you embed Snowflake SQL syntax directly in a Snowpark program

* **DataFrame programming**, which uses DataFrame transformation methods--using a programming style more familiar to users of Python pandas, R, and Spark

The notebook demonstrates both techniques. Finally, the notebook demonstrates saving the flattening work back to Snowflake, as both a database view and as a table.

### Steps below:

1. Create a Snowflake session from saved properties
2. Create a DataFrame from the Snowflake SFO airport weather data
3. Flatten the nested structure (two techniques)
4. Save flattened data

In [2]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session
from snowflake.snowpark.types import *

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

### 1. Create a Snowlake session from saved properties

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], 'rb') as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{'private_key': private_key_bytes}}).create()

### 2. Create a DataFrame from the Snowflake SFO airport weather data

* Define the DataFrame from the SFO_10YR_JSON weather table

In [ ]:
weatherDF = session.table('data_science_db.noaa.sfo_10yr_json')

* Examine the table: Show the table structure, row count, and a sample record

Of course here the term *schema* refers to the structure or description of a DataFrame, not a schema object that defines a namespace in Snowflake.

In [ ]:
weatherDF.schema.fields

In [ ]:
weatherDF.count()

In [ ]:
weatherDF.show(1)

### 3. Flatten the nested structure (two techniques)

#### Option 1: Derive the DataFrame from a SQL SELECT statement
> If the argument to a session.sql() method call is a SELECT statement, then the method defines a DataFrame on the result of the SELECT.

> Note, the SQL statement is placed in triple double quotes in order to accept all quotes and other characters within the statement string.

In [ ]:
sfo_10yr_option1_DF = (
    session.sql("""select distinct
       v:station:id::int        AS station_id,
       v:station:name::string   AS station_name,
       v:station:coord.lat::float  AS lat,
       v:station:coord.lon::float  AS lon,
       value:dt::datetime          AS dt,
       value:air:temp::float       AS air_temp,
       value:air['temp-quality-code']::char(1)        AS temp_quality
    from data_science_db.noaa.sfo_10yr_json,
         lateral flatten(input=>v:data:observations)""")
)

* Examine the DataFrame

In [ ]:
sfo_10yr_option1_DF.schema.fields

In [ ]:
sfo_10yr_option1_DF.count()

In [ ]:
sfo_10yr_option1_DF.sort(col('dt')).show(5)

#### Option 2: Flatten using DataFrame transformations

> Rather than using SQL, you can transform a DataFrame using methods in the [Snowpark API](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/index.html) itself.

> In the syntax below, multilple DataFrame transformations are chained together, so that each line defines a new DataFrame derived from the line just above.

> For a definitive list of the methods available on DataFrames, see the [DataFrame class](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/dataframe.html) in the API documentation.

In [ ]:
sfo_10yr_option2_DF = (
    weatherDF.
    join_table_function('flatten', col('v'),lit('data:observations')).
    withColumn('station_id', col('v')['station']['id'].cast(IntegerType())).
    withColumn('station_name', col('v')['station']['name'].cast(StringType())).
    withColumn('lat', col('v')['station']['coord']['lat'].cast(FloatType())).
    withColumn('lon', col('v')['station']['coord']['lon'].cast(FloatType())).
    withColumn('dt', col('value')['dt'].cast(TimestampType())).
    withColumn('air_temp', col('value')['air']['temp'].cast(FloatType())).
    withColumn('temp_quality', col('value')['air']['temp-quality-code'].cast(StringType())).
    select('station_id', 'station_name', 'lat', 'lon', 'dt', 'air_temp', 'temp_quality').
    distinct()
)

* Examine the DataFrame

In [ ]:
sfo_10yr_option2_DF.schema.fields

In [ ]:
sfo_10yr_option2_DF.count()

In [ ]:
sfo_10yr_option2_DF.sort(col('dt')).show(5)

### 4. Save flattened data (two ways)

* (1) As a Snowflake **view**

In [ ]:
user_db = props['user'] + '_DB'
view_to_create = user_db + '.public.sfo_temps_vw'
sfo_10yr_option2_DF.createOrReplaceView(view_to_create)

* (2) As a Snowflake **table**: Materialize the content of the DataFrame and save as a table in Snowflake.

> The write() method on a DataFrame returns a [DataFrameWriter](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrameWriter.html#snowflake.snowpark.DataFrameWriter) object.

> Prior to writing a table to Snowflake, you can set the **mode** for saving data to 'append', 'overwrite', 'errorifexists', or 'ignore'.

In [ ]:
table_to_save = user_db + '.public.sfo_temps'

(sfo_10yr_option1_DF.
 write.
 mode('overwrite').
 save_as_table(table_to_save)
)